**Natural Language Processng with Transformers**

**Objectives**
* Apply a pre-trained BERT model for sentence encoding using Hugging Face Transformers
* Extract token-level contextual embeddings
* Demonstrate how BERT captures word meaning based on context (polysemy)
* Compute and interpret cosine similarity between token embeddings
* Explain the importance of contextual embeddings in contrast to static word vectors

In [1]:
# Import relevant libraries
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Load pre-trained BERT tokenizer and BERT model from Hugging Face
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = TFBertModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were 

In [12]:
# Example sentence pairs
sentence_pairs = [
    ("How do I learn Python?", "What is the best way to study Python?"),
    ("What is AI?", "How to cook pasta?"),
    ("How do I bake a chocolate cake?", "Give me a chocolate cake recipe."),
    ("How can I improve my coding skills?", "Tips for becoming better at programming."),
    ("Where can I buy cheap laptops?", "Best sites to find affordable computers."),
    ("How do I learn Python quickly?", "What are the fastest ways to pick up Python?"),
    ("What is Artificial Intelligence?", "Can you explain AI in simple terms?"),
    ("How do I bake a delicious chocolate cake?", "Looking for a great chocolate cake recipe."),
    ("How can I improve my programming skills efficiently?", "Effective tips for becoming a better programmer."),
    ("Where can I find affordable laptops online?", "Best websites to buy cheap computers."),
    ("What is machine learning?", "Explain what machine learning is."),
    ("How do I build a website?", "Steps to create a website."),
    ("What are the benefits of cloud computing?", "Advantages of using cloud services."),
    ("How to secure my computer?", "Tips for computer security."),
    ("What is data science?", "Can you define data science?"),
    ("What is the capital of France?", "Can you tell me the capital city of France?"),
    ("How does photosynthesis work?", "Explain the process of photosynthesis."),
    ("What are the benefits of exercise?", "Why is exercise good for you?"),
    ("How to make a simple omelette?", "Easy recipe for an omelette."),
    ("What are the symptoms of the common cold?", "Signs that you have a cold."),
    ("How do I save money?", "Tips for saving cash."),
    ("What is the history of the internet?", "Tell me about the origin of the internet."),
    ("How to write a resume?", "Guide to creating a CV."),
    ("What are the different types of clouds?", "Name some cloud classifications."),
    ("How to train a dog?", "Methods for dog training.")
]

In [13]:
# Ground truth similarity labels: 1 = similar, 0 = not similar
labels = [1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1]
# labels = [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

In [14]:
# Function to get the BERT [CLS] embedding for sentence
def get_bert_embedding(sentence):
  input_ids = tokenizer(sentence, return_tensors="tf", padding=True, truncation=True)
  # input_ids = tokenizer.encode(sentence, add_special_tokens=True)
  # outputs = bert_model(input_ids)
  # return outputs.last_hidden_state[:, 0, :].numpy()
  # Get model output
  outputs = bert_model(input_ids)

  # Extract [CLS] token embedding (shape: [1, 768])
  cls_embedding = outputs.last_hidden_state[:, 0, :].numpy()
  return cls_embedding

In [15]:
# Calculate cosine similarity for each pair
similarities = []
for sent1, sent2 in sentence_pairs:
  emb1 = get_bert_embedding(sent1)
  emb2 = get_bert_embedding(sent2)
  sim_score = cosine_similarity(emb1, emb1)[0][0]
  pred = 1 if sim_score > 0.7 else 0
  similarities.append(sim_score)

  # Print outputs
  print(f"\nSentence 1: {sent1}")
  print(f"Sentence 2: {sent2}")
  print(f"Similarity Score: {sim_score:.4f} -> Predicted Similar: {pred}")


Sentence 1: How do I learn Python?
Sentence 2: What is the best way to study Python?
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: What is AI?
Sentence 2: How to cook pasta?
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: How do I bake a chocolate cake?
Sentence 2: Give me a chocolate cake recipe.
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: How can I improve my coding skills?
Sentence 2: Tips for becoming better at programming.
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: Where can I buy cheap laptops?
Sentence 2: Best sites to find affordable computers.
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: How do I learn Python quickly?
Sentence 2: What are the fastest ways to pick up Python?
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: What is Artificial Intelligence?
Sentence 2: Can you explain AI in simple terms?
Similarity Score: 1.0000 -> Predicted Similar: 1

Sentence 1: How do I bake a de

In [16]:
# Evaluate Accuracy
correct = 0
for i in range(len(similarities)):
  if similarities[i] == labels[i]:
    correct += 1

# Final accuracy calculation
accuracy = correct / len(similarities)
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.28
